# DoshaNet Explain and Evaluate

This notebook loads artifacts from a run folder and writes plots to that folder.

In [1]:
import os, json
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.data import HeteroData
from torch_geometric.nn import HANConv
from sklearn.metrics import confusion_matrix, roc_auc_score, cohen_kappa_score, accuracy_score
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import kneighbors_graph

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

/Users/chinmayabharadwajhs/ml/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_DIR = ''
if not RUN_DIR:
    runs_root = 'runs'
    if os.path.isdir(runs_root):
        candidates = [os.path.join(runs_root, d) for d in os.listdir(runs_root) if d.startswith('exp_')]
        candidates = [d for d in candidates if os.path.isdir(d)]
        RUN_DIR = max(candidates, key=os.path.getmtime) if candidates else ''
print('RUN_DIR:', RUN_DIR)
assert RUN_DIR, 'Set RUN_DIR to your training output folder'

RUN_DIR: runs/exp_20260429_132506


In [3]:
prep_path = os.path.join(RUN_DIR, 'prep_data.npz')
data = np.load(prep_path, allow_pickle=True)
X_full = data['X_full'].astype(np.float32)
y_full = data['y_full'].astype(np.int64)
feature_cols = data['feature_cols'].tolist()
dosha_names = data['dosha_names'].tolist()
train_mask = data['train_mask'].astype(bool)
test_mask = data['test_mask'].astype(bool)
NUM_CLASSES = len(dosha_names)

graph_data = torch.load(os.path.join(RUN_DIR, 'graph_data.pt'), map_location='cpu', weights_only=False)
k_neighbors = graph_data.get('k_neighbors', 10)
print('Classes:', NUM_CLASSES, 'k_neighbors:', k_neighbors)

Classes: 6 k_neighbors: 9


In [6]:
def build_hetero_graph(X, y, feature_cols, dosha_names, k_neighbors=10):
    data = HeteroData()
    num_patients = len(X)
    num_symptoms = len(feature_cols)
    num_doshas = len(dosha_names)

    data['patient'].x = torch.tensor(X, dtype=torch.float)
    data['patient'].y = torch.tensor(y, dtype=torch.long)
    data['symptom'].x = torch.eye(num_symptoms, dtype=torch.float)
    data['dosha'].x = torch.eye(num_doshas, dtype=torch.float)

    medians = np.median(X, axis=0)
    p_idx, s_idx = [], []
    for p in range(num_patients):
        for s in range(num_symptoms):
            if X[p, s] >= medians[s]:
                p_idx.append(p)
                s_idx.append(s)
    data['patient', 'has_trait', 'symptom'].edge_index = torch.tensor([p_idx, s_idx], dtype=torch.long)

    data['patient', 'belongs_to', 'dosha'].edge_index = torch.tensor(
        [list(range(num_patients)), list(y)], dtype=torch.long
    )

    adj = kneighbors_graph(X, n_neighbors=k_neighbors, metric='cosine', include_self=False)
    rows, cols = adj.nonzero()
    data['patient', 'similar_to', 'patient'].edge_index = torch.tensor([rows, cols], dtype=torch.long)
    return data

class HeteroDoshaNet(torch.nn.Module):
    def __init__(self, in_channels_dict, hidden_dim, num_classes, heads=4, dropout=0.3):
        super().__init__()
        self.metadata = (
            ['patient', 'symptom', 'dosha'],
            [('patient', 'has_trait', 'symptom'),
             ('patient', 'belongs_to', 'dosha'),
             ('patient', 'similar_to', 'patient')]
        )
        self.han1 = HANConv(in_channels_dict, hidden_dim, metadata=self.metadata, heads=heads, dropout=dropout)
        self.bn = torch.nn.BatchNorm1d(hidden_dim)
        self.drop = torch.nn.Dropout(dropout)
        self.han2 = HANConv(hidden_dim, num_classes, metadata=self.metadata, heads=1, dropout=dropout)

    def forward(self, x_dict, edge_index_dict):
        out = self.han1(x_dict, edge_index_dict)
        out = {k: F.elu(self.bn(v)) for k, v in out.items()}
        out = {k: self.drop(v) for k, v in out.items()}
        out = self.han2(out, edge_index_dict)
        return F.log_softmax(out['patient'], dim=1)

in_ch = {'patient': X_full.shape[1], 'symptom': len(feature_cols), 'dosha': NUM_CLASSES}
ckpt_path = os.path.join(RUN_DIR, 'best_model.pt')
state_dict = torch.load(ckpt_path, map_location='cpu')

# Infer hidden size and heads from checkpoint to avoid size mismatches
hidden_dim = int(state_dict.get('bn.running_mean', torch.zeros(64)).shape[0])
heads = 4
probe_key = 'han1.lin_src.patient__belongs_to__dosha'
if probe_key in state_dict:
    heads = int(state_dict[probe_key].shape[1])
model = HeteroDoshaNet(in_ch, hidden_dim=hidden_dim, num_classes=NUM_CLASSES, heads=heads, dropout=0.3)
model.load_state_dict(state_dict)
model.eval()

data_obj = build_hetero_graph(X_full, y_full, feature_cols, dosha_names, k_neighbors=k_neighbors)

/var/folders/df/yfx9tpqs7qq6bc8ycgcb202r0000gn/T/ipykernel_46216/945127855.py:27: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  data['patient', 'similar_to', 'patient'].edge_index = torch.tensor([rows, cols], dtype=torch.long)


In [7]:
def explain_prediction(model, data, patient_idx, feature_cols, dosha_names, out_dir):
    model.eval()
    data['patient'].x.requires_grad = True

    out = model(data.x_dict, data.edge_index_dict)
    probs = torch.exp(out[patient_idx])
    pred_class = probs.argmax().item()
    confidence = probs.max().item() * 100

    out[patient_idx, pred_class].backward()
    feat_importance = data['patient'].x.grad[patient_idx].abs().cpu().numpy()
    top5_idx = np.argsort(feat_importance)[-5:][::-1]

    print('Patient', patient_idx, '->', dosha_names[pred_class], f'({confidence:.1f}%)')
    for idx in top5_idx:
        print('  ', feature_cols[idx], 'importance:', round(float(feat_importance[idx]), 6))

    colors = ['#3fb950' if i in top5_idx else '#6e7681' for i in range(len(feature_cols))]
    plt.figure(figsize=(12, 5))
    plt.bar(range(len(feature_cols)), feat_importance, color=colors)
    plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha='right', fontsize=8)
    plt.title(f'GNNExplainer - Feature Attribution (Patient {patient_idx})')
    plt.tight_layout()
    out_path = os.path.join(out_dir, f'explanation_patient_{patient_idx}.png')
    plt.savefig(out_path, dpi=150)
    plt.close()

    data['patient'].x.requires_grad = False
    return feat_importance, pred_class

test_indices = np.where(test_mask)[0]
for idx in test_indices[:3]:
    explain_prediction(model, data_obj, idx, feature_cols, dosha_names, RUN_DIR)

Patient 959 -> pitta (98.4%)
   Cheeks importance: 0.012193
   Complexion importance: 0.008016
   Hair Color importance: 0.006998
   Body Size importance: 0.006982
   Stress Levels importance: 0.006784
Patient 960 -> vata+pitta (100.0%)
   Dietary Habits importance: 2e-06
   Nails importance: 2e-06
   Teeth and gums importance: 2e-06
   Metabolism Type importance: 1e-06
   Appearance of Hair importance: 1e-06
Patient 961 -> pitta (100.0%)
   Eyelashes importance: 0.0
   Body Weight importance: 0.0
   Cheeks importance: 0.0
   Lips importance: 0.0
   Physical Activity Level importance: 0.0


In [8]:
with torch.no_grad():
    out = model(data_obj.x_dict, data_obj.edge_index_dict)
preds = out[test_mask].argmax(dim=1).cpu().numpy()
y_true = y_full[test_mask]

cm = confusion_matrix(y_true, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=dosha_names, yticklabels=dosha_names)
plt.title('Confusion Matrix - DoshaNet HAN')
plt.tight_layout()
cm_path = os.path.join(RUN_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.close()

acc = accuracy_score(y_true, preds)
print('Test accuracy:', round(acc * 100, 2), '%')
print('Saved:', cm_path)

Test accuracy: 98.33 %
Saved: runs/exp_20260429_132506/confusion_matrix.png


In [9]:
def predict_with_uncertainty(model, data, n_samples=50):
    model.train()
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            out = model(data.x_dict, data.edge_index_dict)
            preds.append(torch.exp(out).unsqueeze(0))

    preds = torch.cat(preds, dim=0)
    mean = preds.mean(dim=0)
    variance = preds.var(dim=0)
    entropy = -(mean * torch.log(mean + 1e-8)).sum(dim=1)

    predicted = mean.argmax(dim=1)
    confidence = mean.max(dim=1).values * 100
    return predicted, confidence, entropy, variance

pred_d, conf, entropy, _ = predict_with_uncertainty(model, data_obj, n_samples=50)
for i in np.where(test_mask)[0][:5]:
    label = 'High' if conf[i] > 80 and entropy[i] < 0.5 else ('Moderate' if conf[i] > 60 else 'Low')
    print('Patient', i, ':', dosha_names[pred_d[i]], '| Conf', round(float(conf[i]), 1), '% | Ent', round(float(entropy[i]), 3), '|', label)

Patient 959 : pitta | Conf 98.3 % | Ent 0.108 | High
Patient 960 : vata+pitta | Conf 98.4 % | Ent 0.109 | High
Patient 961 : pitta | Conf 100.0 % | Ent 0.005 | High
Patient 962 : pitta | Conf 99.6 % | Ent 0.032 | High
Patient 963 : pitta+kapha | Conf 99.6 % | Ent 0.031 | High


In [10]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_accs = []
for fold, (tr, te) in enumerate(kf.split(X_full, y_full)):
    clf = RandomForestClassifier(n_estimators=200, random_state=SEED)
    clf.fit(X_full[tr], y_full[tr])
    acc = clf.score(X_full[te], y_full[te])
    fold_accs.append(acc)
    print('Fold', fold + 1, ':', round(acc * 100, 2), '%')
cv_mean = float(np.mean(fold_accs))
cv_std = float(np.std(fold_accs))
print('CV mean:', round(cv_mean * 100, 2), 'std:', round(cv_std * 100, 2))

rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
rf.fit(X_full[train_mask], y_full[train_mask])
y_prob = rf.predict_proba(X_full[test_mask])
y_bin = label_binarize(y_full[test_mask], classes=list(range(NUM_CLASSES)))
auc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
print('Macro ROC-AUC:', round(float(auc), 4))

kappa = cohen_kappa_score(y_true, preds)
print('Cohen kappa:', round(float(kappa), 4))

mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=500, random_state=SEED)
mlp.fit(X_full[train_mask], y_full[train_mask])
gnn_acc = accuracy_score(y_true, preds)
print('Ablation:')
print('  MLP:', round(mlp.score(X_full[test_mask], y_full[test_mask]) * 100, 2), '%')
print('  RF:', round(rf.score(X_full[test_mask], y_full[test_mask]) * 100, 2), '%')
print('  HAN:', round(gnn_acc * 100, 2), '%')

Fold 1 : 100.0 %
Fold 2 : 100.0 %
Fold 3 : 100.0 %
Fold 4 : 100.0 %
Fold 5 : 100.0 %
CV mean: 100.0 std: 0.0
Macro ROC-AUC: 1.0
Cohen kappa: 0.9745
Ablation:
  MLP: 100.0 %
  RF: 100.0 %
  HAN: 98.33 %


In [11]:
model_card = {
    'model_name': 'DoshaNet - Heterogeneous Graph Attention Network',
    'architecture': {
        'type': 'HANConv (Heterogeneous Attention Network)',
        'node_types': ['patient', 'symptom', 'dosha'],
        'edge_types': ['has_trait', 'belongs_to', 'similar_to'],
        'layers': 2,
        'attention_heads': 4,
        'parameters': sum(p.numel() for p in model.parameters())
    },
    'training': {
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'hyperparams': None,
        'epochs': None,
        'early_stopping': True
    },
    'verification': {
        'test_accuracy': round(gnn_acc * 100, 2),
        'cv_mean': round(cv_mean * 100, 2),
        'cv_std': round(cv_std * 100, 2),
        'macro_roc_auc': round(float(auc), 4),
        'cohens_kappa': round(float(kappa), 4)
    }
}

card_path = os.path.join(RUN_DIR, 'model_card.json')
with open(card_path, 'w') as f:
    json.dump(model_card, f, indent=2)
print('Saved:', card_path)

Saved: runs/exp_20260429_132506/model_card.json
